In [1]:
import boto3
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.pipelines.inference import run_summary, get_fresh_claims, get_retrieved_claims

In [2]:
user_query = "How does caffeine influence cellular energy metabolism and mitochondrial function?"
retrieved = get_retrieved_claims(user_query, 2)
retrieved

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::008',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH+TRE group Evidence: ['PSQI before and after intervention in DASH+TRE group, n=37:37, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH+TRE group',
   'evidence': ['PSQI before and after intervention in DASH+TRE group, n=37:37, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small

In [3]:
fresh = get_fresh_claims(user_query, 10)

 Extracted keywords: caffeine, cellular energy metabolism, mitochondrial function
Number of papers: 10
Papers
: [{'id': '65116f5fe4d30002ab089c94df941e6e972938e5', 'title': 'Mitochondrial calcium uniporter in Drosophila transfers calcium between the endoplasmic reticulum and mitochondria in oxidative stress-induced cell death', 'year': 2017, 'venue': 'Journal of Biological Chemistry', 'fieldsOfStudy': ['Biology', 'Medicine'], 'publicationTypes': ['JournalArticle'], 'pdf_url': 'https://europepmc.org/articles/pmc5582840?pdf=render'}, {'id': '7098c3df4e756545b09bb73b86e6dc5788402d72', 'title': 'Cardiac lipid metabolism, mitochondrial function and heart failure.', 'year': 2023, 'venue': 'Cardiovascular Research', 'fieldsOfStudy': ['Medicine'], 'publicationTypes': ['JournalArticle', 'Review'], 'pdf_url': 'https://academic.oup.com/cardiovascres/advance-article-pdf/doi/10.1093/cvr/cvad100/50887555/cvad100.pdf'}, {'id': '745ccc1e7452e6923b93168c1c8c886dbb4bfda4', 'title': 'Mitochondrial Dysfun

In [4]:
fresh

[{'id': '65116f5fe4d30002ab089c94df941e6e972938e5::001',
  'embedding_text': "Claim: Drosophila MCU loss-of-function mutants lacked mitochondrial calcium uptake in response to caffeine stimulation Evidence: ['MCU52 mutant failed to elicit any [Ca2+]mito change upon caffeine stimulation (Fig. 2, D and E)']",
  'metadata': {'paper_id': '65116f5fe4d30002ab089c94df941e6e972938e5',
   'claim': 'Drosophila MCU loss-of-function mutants lacked mitochondrial calcium uptake in response to caffeine stimulation',
   'evidence': ['MCU52 mutant failed to elicit any [Ca2+]mito change upon caffeine stimulation (Fig. 2, D and E)'],
   'section': ['Results'],
   'study_quality': {'study_design': 'observational',
    'sample_size': 'unclear',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'The study employs a loss-of-function approach using Drosophila mutants (MCU52) combined with cell culture experiments (S2 cells) and transgenic rescue experiments. This is a well-establish

In [5]:
summarized = run_summary(retrieved, fresh)

In [6]:
summarized

{'summary': [{'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in either the DASH group (n=26, p>0.05) or the DASH+TRE group (n=37, p>0.05).',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
    '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::008'],
   'evidence_strength': 'strong'}],
 'confidence_note': 'Evidence is derived from a randomized controlled trial with modest sample sizes (total n=63). While evidence strength is rated as strong due to specific numerical outcomes with statistical significance testing, the study carries moderate bias risk due to limited sample size, unclear blinding status, and no mention of allocation concealment or loss to follow-up reporting.'}

In [ ]:
from src.agents.counter_argument import CounterArgumentAgent
from src.preprocessing.normalize_similarity import normalize_similarity
from src.preprocessing.clean_json import clean_json

In [10]:
counterarg = CounterArgumentAgent()

In [11]:
fresh_claims_sim_scored = [normalize_similarity(c) for c in fresh]
all_claims = retrieved + fresh_claims_sim_scored


In [13]:
selected_claims = select_counter_claims(all_claims)
counter_res = counterarg.run_counterarg(selected_claims)
counter_res = clean_json(counter_res)

In [14]:
counter_res

{'limitations': [{'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant improvement in sleep quality following DASH dietary intervention, with p >0.05 indicating null results.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007'],
   'evidence_strength': 'strong'},
  {'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant improvement in sleep quality following DASH+TRE (time-restricted eating) intervention, with p >0.05 indicating null results.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::008'],
   'evidence_strength': 'strong'},
  {'statement': 'The study examining DASH and DASH+TRE interventions had a small sample size (total n=63) and moderate bias risk due to unclear blinding status, no mention of allocation concealment or loss to follow-up reporting, which limits generalizability of the null sleep quality findings.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007

In [15]:
def build_final_response(summary_output, counter_output):
    key_findings = summary_output["summary"]
    limitations = counter_output.get("limitations", [])

    citation_ids = set()
    for item in key_findings + limitations:
        citation_ids.update(item.get("supporting_claim_ids", []))

    return {
        "query":user_query,
        "answer": {
            "key_findings": key_findings,
            "confidence_note": summary_output.get("confidence_note", "")
        },
        "limitations": limitations,
        "metadata": {
            "num_supporting_claims": len(citation_ids),
            "num_limitations": len(limitations),
            "citation_ids": sorted(citation_ids)
        }
    }

In [16]:
final = build_final_response(summarized, counter_res)
final

{'query': 'How does caffeine influence cellular energy metabolism and mitochondrial function?',
 'answer': {'key_findings': [{'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in either the DASH group (n=26, p>0.05) or the DASH+TRE group (n=37, p>0.05).',
    'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
     '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::008'],
    'evidence_strength': 'strong'}],
  'confidence_note': 'Evidence is derived from a randomized controlled trial with modest sample sizes (total n=63). While evidence strength is rated as strong due to specific numerical outcomes with statistical significance testing, the study carries moderate bias risk due to limited sample size, unclear blinding status, and no mention of allocation concealment or loss to follow-up reporting.'},
 'limitations': [{'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant improvement in sleep q

In [ ]:
# upload index to s3 finally

s3 = session.client("s3", region_name="ca-central-1")
def upload_file(local_path: str, bucket: str, s3_key: str):
    s3.upload_file(local_path, bucket, s3_key)
    print(f"Uploaded {s3_key}")

upload_file(INDEX_PATH, BUCKET, "faiss/claims.index")
upload_file(META_PATH, BUCKET, "faiss/claims_meta.pkl")